# Week8 Jetson 接收与抓取

在 Jetson 的 JupyterLab 中运行本 notebook。只上传这一个 notebook 即可获得业务代码；仍需设备原有 SDK/config 和 NumPy/OpenCV/pyserial/ipywidgets。关闭其他 Week6/Week7 的相机及串口连接。启动服务只连接设备，不移动；本机发送 prepare 后才回拍摄位，1/2/3/5 加 OK 后才抓取。


## 集中参数

手势默认 1红、2黄、3绿、5蓝，OK 确认。Five 不再停止。接收服务使用 8766 端口。下方抓取参数取自本次提供的 Week6 代码，原文件中的旧文字说明不参与计算。


In [ ]:
"""Week8 parameters extracted from the supplied Week6 code, not its outdated prose."""
SORT_CONFIG = {'sdk_dir': '/home/jetson/Notebook/4.机械臂OpenCV视觉处理实验',
 'arm_port': '/dev/ttyTHS1',
 'camera_device': 0,
 'roi': (240, 210, 440, 390),
 'min_area': 700,
 'move_seconds': 1.5,
 'gripper_seconds': 0.5,
 'p4_pick': (200.2, -12.8, 45.0, 132.7, 179.0),
 'p4_above': (198.2, -12.8, 75.0, 132.7, 179.0),
 'bin_map': {'red': {'bin': '有害垃圾', 'pose': (103.3, -132.5, 175.1, 104.0, -945200.4)},
             'yellow': {'bin': '其他垃圾', 'pose': (219.0, 99.8, 175.6, 100.0, 137329.9)},
             'green': {'bin': '厨余垃圾', 'pose': (122.8, 114.8, 172.8, 104.0, -945200.4)},
             'blue': {'bin': '可回收垃圾', 'pose': (207.8, -141.3, 173.3, 99.5, 137329.9)}},
 'hsv': {'red': [((0, 80, 50), (10, 255, 255)), ((170, 80, 50), (179, 255, 255))],
         'yellow': [((18, 80, 60), (38, 255, 255))],
         'green': [((40, 60, 35), (90, 255, 255))],
         'blue': [((90, 75, 45), (135, 255, 255))]},
 'empty_frames': 8,
 'post_return_settle': 0.6,
 'p4_clear_timeout': 3.0,
 'min_scene_brightness': 10,
 'frame_timeout': 1.0}
GESTURE_COLOR_MAP = {'One':'red','Two':'yellow','Three':'green','Five':'blue'}
FIVE_MODEL_THRESHOLD = 0.50
FIVE_EXTEND_RATIO = 1.55
FIVE_CONFIRM_FRAMES = 2
OTHER_GESTURE_THRESHOLD = 0.70
OTHER_CONFIRM_FRAMES = 3
RECEIVER_PORT = 8766
API_VERSION = 'week8-v1'


## 内置 arm_interrupt

直接运行，不需要另存为 .py 文件。


In [ ]:
"""Cooperative trajectory cancellation plus feedback-based position hold.

Only the motion worker touches the SDK/UART. The HTTP stop handler sets an Event.
This is a software protective stop, not a certified hardware emergency stop.
"""
import math
from pathlib import Path
import sys
import threading
import time

class Cancelled(Exception):
    pass

class InterruptibleArm:

    def __init__(self, sdk_dir, port, stop_event):
        sys.path.insert(0, str(Path(sdk_dir).resolve()))
        from arm5dof_uservo import Arm5DoFUServo
        from uservo import Packet
        self.stop_event = stop_event
        self.arm = Arm5DoFUServo.__new__(Arm5DoFUServo)
        try:
            Arm5DoFUServo.__init__(self.arm, device=port, is_init_pose=False, config_folder=str(Path(sdk_dir).resolve() / 'config'))
        except (Exception, SystemExit) as error:
            uart = getattr(self.arm, 'uart', None)
            if uart is not None:
                uart.close()
            raise RuntimeError('机械臂初始化失败；检查端口、舵机在线和配置：' + str(error)) from error
        self.manager = self.arm.uservo
        self.ids = sorted(self.manager.servos)
        if self.ids != list(range(6)):
            self.arm.uart.close()
            raise RuntimeError('Expected six online servos with IDs 0..5; found ' + str(self.ids))
        self.allow_hold = False
        self.feedback = {identifier: 0 for identifier in self.ids}
        for code in (self.manager.CODE_QUERY_SERVO_ANGLE, self.manager.CODE_QUERY_SERVO_ANGLE_MTURN):
            original = self.manager.response_handle_funcs[code]

            def response(data, original=original):
                original(data)
                self.feedback[data[0]] = self.feedback.get(data[0], 0) + 1
            self.manager.response_handle_funcs[code] = response
        motion_codes = {getattr(self.manager, name) for name in dir(self.manager) if name.startswith('CODE_SET_SERVO_ANGLE')}

        def send(code, data):
            if code in motion_codes and (not self.allow_hold):
                self.check()
            packet = Packet.pack(code, data)
            if self.arm.uart.write(packet) != len(packet):
                raise IOError('Incomplete servo UART write')
        self.manager.send_request = send
        self.arm.uart.write_timeout = 0.2
        self.original_query = self.manager.query_servo_angle
        self.manager.query_servo_angle = self.query
        self.arm.trajectory_callback_period = 1

    def check(self):
        if self.stop_event.is_set():
            raise Cancelled('Motion cancelled')

    def query(self, identifier):
        previous = self.feedback[identifier]
        angle = self.original_query(identifier)
        if self.feedback[identifier] == previous or angle is None or (not math.isfinite(float(angle))):
            raise IOError('Missing fresh angle feedback from servo ' + str(identifier))
        return float(angle)

    def execute(self, command):
        self.check()
        if command == 'open_gripper':
            self.arm.gripper_open(callback=self.check)
        elif command == 'close_gripper':
            self.arm.gripper_close(callback=self.check)
        elif command == 'home':
            result = self.arm.set_tool_pose(pose_name='home', T=2.0, is_soft_move=True, callback=self.check)
            if result is False:
                raise RuntimeError('Home inverse kinematics failed')
        else:
            raise ValueError('Unknown action')
        deadline = time.monotonic() + 5
        while time.monotonic() < deadline:
            self.check()
            if self.manager.is_stop():
                return
            self.stop_event.wait(0.02)
        raise TimeoutError('Servo did not reach target')

    def hold(self):
        """Override the unfinished target with each servo's freshly measured angle."""
        self.allow_hold = True
        try:
            errors = []
            targets = {}
            for identifier in self.ids:
                try:
                    angle = self.query(identifier)
                    targets[identifier] = angle
                    if self.manager.set_servo_angle(identifier, angle, interval=0, power=0) is False:
                        raise IOError('Servo rejected hold')
                except Exception as error:
                    errors.append(str(error))
            if errors:
                raise RuntimeError('; '.join(errors))
            deadline = time.monotonic() + 1.5
            stable = 0
            previous = {identifier: self.query(identifier) for identifier in self.ids}
            while time.monotonic() < deadline:
                time.sleep(0.05)
                current = {identifier: self.query(identifier) for identifier in self.ids}
                near_target = all((abs(current[i] - targets[i]) <= 1.0 for i in self.ids))
                stable = stable + 1 if near_target and all((abs(current[i] - previous[i]) <= 0.5 for i in self.ids)) else 0
                if stable >= 2:
                    return {'hold_sent': True, 'feedback_stable': True}
                previous = current
            raise RuntimeError('Hold sent but stationary feedback not confirmed')
        finally:
            self.allow_hold = False

    def close(self):
        self.arm.uart.close()


## 内置 vision_sorter

直接运行，不需要另存为 .py 文件。


In [ ]:
"""Jetson color verification and Week6's supplied fixed-P4 pick sequence."""
from datetime import datetime
import copy
import json
import math
from pathlib import Path
import sys
import threading
import time
import cv2
import numpy as np
COLORS = ('red', 'yellow', 'green', 'blue')
BGR = {'red': (0, 0, 255), 'yellow': (0, 255, 255), 'green': (0, 255, 0), 'blue': (255, 0, 0)}

def detect_color(frame, config):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = config['roi'] if config['roi'] is not None else (0, 0, w, h)
    if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
        raise ValueError('P4 ROI 超出实际相机画面；格式 x1,y1,x2,y2')
    crop = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    kernel = np.ones((5, 5), np.uint8)
    objects = []
    canvas = frame.copy()
    for color in COLORS:
        mask = np.zeros(crop.shape[:2], np.uint8)
        for lower, upper in config['hsv'][color]:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.asarray(lower, np.uint8), np.asarray(upper, np.uint8)))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        for contour in cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]:
            area = float(cv2.contourArea(contour))
            if area < config['min_area']:
                continue
            x, y, bw, bh = cv2.boundingRect(contour)
            target = dict(color=color, pixel=(x + x1 + bw // 2, y + y1 + bh // 2), area=area)
            objects.append(target)
            cv2.rectangle(canvas, (x + x1, y + y1), (x + x1 + bw, y + y1 + bh), BGR[color], 2)
            cv2.putText(canvas, color, (x + x1, max(20, y + y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, BGR[color], 2)
    cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 255, 255), 1)
    usable = float(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).mean()) >= config['min_scene_brightness']
    return dict(objects=objects, usable=usable, canvas=canvas)

class CameraMonitor:

    def __init__(self, config):
        self.config = config
        self.cv = threading.Condition(threading.RLock())
        self.stop_event = threading.Event()
        self.latest = None
        self.error = None
        self.sequence = 0
        self.need_empty = False
        self.empty_count = 0
        self.empty_after = 0
        self.thread = threading.Thread(target=self.read, daemon=True)

    def start(self):
        self.thread.start()
        with self.cv:
            self.cv.wait_for(lambda: self.latest is not None or self.error is not None, timeout=3)
        self.health()

    def read(self):
        cap = cv2.VideoCapture(self.config['camera_device'])
        try:
            if not cap.isOpened():
                raise RuntimeError('Jetson 摄像头无法打开')
            while not self.stop_event.is_set():
                ok, raw = cap.read()
                if not ok or raw is None:
                    raise RuntimeError('Jetson 摄像头取帧失败')
                result = detect_color(raw, self.config)
                with self.cv:
                    self.sequence += 1
                    self.latest = dict(result, raw=raw.copy(), time=time.monotonic(), sequence=self.sequence)
                    if self.need_empty and self.latest['time'] >= self.empty_after:
                        self.empty_count = self.empty_count + 1 if result['usable'] and (not result['objects']) else 0
                        if self.empty_count >= self.config['empty_frames']:
                            self.need_empty = False
                    self.cv.notify_all()
        except Exception as exc:
            with self.cv:
                self.error = type(exc).__name__ + ': ' + str(exc)
                self.cv.notify_all()
        finally:
            cap.release()

    def health(self):
        with self.cv:
            if self.error:
                raise RuntimeError(self.error)
            if self.latest is None or time.monotonic() - self.latest['time'] > self.config['frame_timeout']:
                raise RuntimeError('Jetson 相机没有新鲜画面')

    def next_after(self, cutoff, cancel, timeout=2):
        deadline = time.monotonic() + timeout
        while time.monotonic() < deadline:
            if cancel.is_set():
                raise Cancelled('Cancelled while waiting for camera')
            with self.cv:
                self.cv.wait(timeout=0.04)
                self.health()
                if self.latest['time'] > cutoff:
                    return dict(self.latest)
        raise RuntimeError('等待 Jetson 新帧超时')

    def require_empty(self, cutoff):
        with self.cv:
            self.need_empty = True
            self.empty_count = 0
            self.empty_after = cutoff

    def state(self):
        with self.cv:
            return dict(camera_error=self.error, camera_fresh=self.latest is not None and time.monotonic() - self.latest['time'] <= self.config['frame_timeout'], p4_colors=[] if self.latest is None else [obj['color'] for obj in self.latest['objects']], p4_need_empty=self.need_empty, empty_frames=self.empty_count)

    def jpeg(self):
        with self.cv:
            if self.latest is None:
                return None
            image = self.latest['canvas'].copy()
        ok, encoded = cv2.imencode('.jpg', image)
        return encoded.tobytes() if ok else None

    def close(self):
        self.stop_event.set()
        self.thread.join(2)
        if self.thread.is_alive():
            raise RuntimeError('相机读取仍未退出，勿重复启动')

def sdk_poses(config):
    sdk = Path(config['sdk_dir']).resolve()
    if not (sdk / 'arm5dof_kinematic.py').is_file():
        raise FileNotFoundError('未找到课程 SDK：' + str(sdk))
    sys.path.insert(0, str(sdk))
    from config import ARM_POSE_DICT, PITCH_PANEL_A, PITCH_PANEL_B, PITCH_PANEL_C
    from arm5dof_kinematic import Arm5DoFKinematic
    k, b = np.loadtxt(str(sdk / 'config/z_error_polyfit.txt'), delimiter=',')

    def from_values(values):
        x, y, z, pitch, roll = values
        if not all((math.isfinite(float(v)) for v in values)):
            raise ValueError('位姿含非法数值')
        return ([float(x), float(y), float(z)], math.radians(pitch), math.radians(roll % 360))
    poses = {'pick': from_values(config['p4_pick']), 'above': from_values(config['p4_above'])}
    for name in ('home', 'capture_image'):
        x, y, z, pitch, roll = ARM_POSE_DICT[name]
        adjusted = z + float(k) * math.hypot(x, y) + float(b)
        if pitch is None:
            pitch = PITCH_PANEL_A * math.hypot(x, y) + PITCH_PANEL_B * adjusted + PITCH_PANEL_C
        poses[name] = ([float(x), float(y), float(z)], pitch, roll)
    for color in COLORS:
        poses[color] = from_values(config['bin_map'][color]['pose'])
    solver = Arm5DoFKinematic()
    for name, (xyz, pitch, roll) in poses.items():
        x, y, z = xyz
        adjusted = z + float(k) * math.hypot(x, y) + float(b)
        if not solver.inverse_kinematic(x, y, adjusted, pitch, roll):
            raise ValueError('IK 无解，未连接串口：' + name + ' ' + str(xyz))
    return poses

class Week8Arm:

    def __init__(self, cancel, config=None):
        self.cancel = cancel
        self.config = copy.deepcopy(SORT_CONFIG if config is None else config)
        self.poses = sdk_poses(self.config)
        self.robot = InterruptibleArm(self.config['sdk_dir'], self.config['arm_port'], cancel)
        self.camera = CameraMonitor(self.config)
        self.view_ready = False
        self.stage = 'CONNECTED_NOT_MOVED'
        self.last_outcome = None
        self.output = Path.cwd() / 'week8_records' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        try:
            self.camera.start()
        except Exception:
            self.camera.stop_event.set()
            self.camera.thread.join(2)
            self.robot.close()
            raise

    def status(self):
        return dict(self.camera.state(), view_ready=self.view_ready, stage=self.stage, last_outcome=self.last_outcome)

    def validate_command(self, command):
        self.camera.health()
        if command == 'prepare':
            return
        if command not in ('sort_red', 'sort_yellow', 'sort_green', 'sort_blue'):
            raise ValueError('未知 Week8 指令')
        if not self.view_ready:
            raise ValueError('先发送“Jetson 回拍摄位”指令')
        if self.camera.state()['p4_need_empty']:
            raise ValueError('上一轮 P4 尚未连续清空；不能重复抓取')

    def log(self, event, **fields):
        self.output.mkdir(parents=True, exist_ok=True)
        with (self.output / 'events.jsonl').open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(dict(time=datetime.now().isoformat(), event=event, stage=self.stage, **fields), ensure_ascii=False) + '\n')

    def check(self):
        self.robot.check()
        self.camera.health()

    def settled(self):
        deadline = time.monotonic() + 5
        while time.monotonic() < deadline:
            self.check()
            if self.robot.manager.is_stop():
                return
            self.cancel.wait(0.02)
        raise RuntimeError('等待关节到位超时')

    def move(self, name):
        self.stage = name
        self.check()
        xyz, pitch, roll = self.poses[name]
        result = self.robot.arm.set_tool_pose(xyz, tool_pitch=pitch, tool_roll=roll, T=self.config['move_seconds'], is_soft_move=True, is_wait=True, callback=self.check)
        if result is False:
            raise RuntimeError(name + ' 运动 IK 失败')
        self.settled()

    def gripper(self, close):
        self.stage = 'GRIP_CLOSE' if close else 'GRIP_OPEN'
        self.check()
        method = self.robot.arm.gripper_close if close else self.robot.arm.gripper_open
        method(T=self.config['gripper_seconds'], is_wait=True, callback=self.check)
        self.settled()

    def execute(self, command):
        self.validate_command(command)
        if command == 'prepare':
            self.view_ready = False
            self.move('capture_image')
            cutoff = time.monotonic() + self.config['post_return_settle']
            if self.camera.state()['p4_need_empty']:
                self.camera.require_empty(cutoff)
            self.camera.next_after(cutoff, self.cancel, timeout=3)
            self.view_ready = True
            self.stage = 'READY_FOR_GESTURE'
            return dict(prepared=True, moved='capture_image')
        color = command[len('sort_'):]
        self.stage = 'VERIFY_SELECTED_' + color.upper()
        frame = self.camera.next_after(time.monotonic(), self.cancel)
        objects = frame['objects']
        if not frame['usable'] or len(objects) != 1:
            raise ValueError('P4 必须恰好有一个有效色块，当前数量 ' + str(len(objects)))
        if objects[0]['color'] != color:
            raise ValueError('手势选择 ' + color + '，但 P4 识别到 ' + objects[0]['color'] + '；未抓取')
        self.output.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime('%H%M%S_%f')
        ok, encoded = cv2.imencode('.png', frame['canvas'])
        if not ok:
            raise IOError('识别截图编码失败')
        filename = stamp + '_' + color + '.png'
        encoded.tofile(str(self.output / filename))
        self.log('sort_start', selected_color=color, detected_color=objects[0]['color'], screenshot=filename)
        self.view_ready = False
        self.camera.require_empty(float('inf'))
        self.gripper(False)
        self.move('above')
        self.move('pick')
        self.gripper(True)
        self.move('above')
        self.move('home')
        self.move(color)
        self.gripper(False)
        self.move('home')
        self.move('capture_image')
        returned = time.monotonic()
        self.camera.require_empty(returned + self.config['post_return_settle'])
        self.stage = 'VERIFY_P4_CLEAR'
        deadline = time.monotonic() + self.config['p4_clear_timeout']
        while time.monotonic() < deadline and self.camera.state()['p4_need_empty']:
            self.check()
            self.cancel.wait(0.04)
        self.view_ready = True
        clear = not self.camera.state()['p4_need_empty']
        self.last_outcome = dict(selected_color=color, detected_color=objects[0]['color'], target_bin=self.config['bin_map'][color]['bin'], commands_returned=True, p4_clear=clear, actual_bin_observed=False, screenshot=filename, message='P4 已清空，桶内结果待人工确认' if clear else '动作已返回但 P4 未清空；检查抓取并移走物块，不会重复抓取')
        self.stage = 'READY_FOR_GESTURE' if clear else 'WAIT_P4_CLEAR'
        self.log('sort_returned', outcome=self.last_outcome)
        return self.last_outcome

    def hold(self):
        self.view_ready = False
        self.camera.require_empty(float('inf'))
        self.stage = 'HOLDING'
        report = self.robot.hold()
        self.stage = 'STOPPED_HELD'
        return report

    def close(self):
        try:
            self.camera.close()
        finally:
            self.robot.close()


## 内置 receiver

直接运行，不需要另存为 .py 文件。


In [ ]:
"""Jetson HTTP receiver over the Micro-USB gadget network (Python 3.6+)."""
import argparse
from collections import OrderedDict
import json
from http.server import BaseHTTPRequestHandler, HTTPServer
try:
    from http.server import ThreadingHTTPServer
except ImportError:
    from socketserver import ThreadingMixIn

    class ThreadingHTTPServer(ThreadingMixIn, HTTPServer):
        daemon_threads = True
import secrets
import threading
import time
import uuid

class Engine:

    def __init__(self, hardware=None, heartbeat_timeout=1.5):
        self.cancel = threading.Event()
        self.cancel.set()
        self.arm = (hardware or Week8Arm)(self.cancel)
        self.mode = 'hardware'
        self.cv = threading.Condition(threading.RLock())
        self.session = uuid.uuid4().hex
        self.epoch = 0
        self.stopped, self.busy = (True, False)
        self.phase = 'locked'
        self.last_seen = None
        self.timeout = heartbeat_timeout
        self.job = None
        self.hold_pending = False
        self.results = OrderedDict()
        self.hold_result = None
        self.shutdown = False
        self.thread = threading.Thread(target=self.worker, daemon=True)
        self.thread.start()
        self.watchdog = threading.Thread(target=self.watch, daemon=True)
        self.watchdog.start()

    def status(self):
        with self.cv:
            vision = self.arm.status()
            return dict(session=self.session, epoch=self.epoch, mode=self.mode, busy=self.busy, stopped=self.stopped, stop_phase=self.phase, hold_result=self.hold_result, api_version=API_VERSION, gesture_color_map=GESTURE_COLOR_MAP, vision=vision)

    def heartbeat(self):
        with self.cv:
            self.last_seen = time.monotonic()
            return self.status()

    def stop(self, reason='Manual stop'):
        with self.cv:
            self.cancel.set()
            if not self.stopped or self.phase == 'locked':
                self.epoch += 1
                self.stopped, self.busy = (True, True)
                self.phase = 'stopping'
                self.hold_pending = True
                self.hold_result = {'reason': reason}
                self.cv.notify_all()
            return self.status()

    def clear(self, epoch):
        with self.cv:
            self.arm.camera.health()
            if epoch != self.epoch or self.busy or self.phase in ('stopping', 'failed'):
                raise ValueError('Busy, stale epoch or stop failed; inspect the arm before reset')
            if self.last_seen is None or time.monotonic() - self.last_seen > self.timeout:
                raise ValueError('No fresh heartbeat')
            self.stopped, self.phase = (False, 'ready')
            self.cancel.clear()
            return self.status()

    def submit(self, identifier, command, epoch):
        with self.cv:
            if identifier in self.results:
                if self.results[identifier]['command'] != command:
                    raise ValueError('同一指令编号不能用于不同动作')
                return dict(self.results[identifier])
            if command not in ('prepare', 'sort_red', 'sort_yellow', 'sort_green', 'sort_blue'):
                raise ValueError('Unknown command')
            if epoch != self.epoch or self.stopped or self.busy:
                raise ValueError('Stopped, busy or obsolete command epoch')
            if self.last_seen is None or time.monotonic() - self.last_seen > self.timeout:
                raise ValueError('Heartbeat expired')
            self.arm.validate_command(command)
            self.busy = True
            self.results[identifier] = dict(id=identifier, status='running', command=command, mode=self.mode)
            while len(self.results) > 1000:
                self.results.popitem(last=False)
            self.job = (identifier, command)
            self.cv.notify_all()
            return dict(self.results[identifier])

    def worker(self):
        while True:
            with self.cv:
                self.cv.wait_for(lambda: self.job is not None or self.hold_pending or self.shutdown)
                if self.shutdown and (not self.job) and (not self.hold_pending):
                    return
                job, self.job = (self.job, None)
            error = None
            outcome = None
            if job:
                try:
                    outcome = self.arm.execute(job[1])
                except Cancelled:
                    self.cancel.set()
                except Exception as exc:
                    error = type(exc).__name__ + ' at ' + self.arm.stage + ': ' + str(exc)
                    try:
                        self.arm.log('execution_error', command=job[1], error=error)
                    except Exception:
                        pass
                    self.stop('Motion failure: ' + error)
            with self.cv:
                needs_hold = self.cancel.is_set()
            hold_error, report = (None, None)
            if needs_hold:
                try:
                    report = self.arm.hold()
                except Exception as exc:
                    hold_error = str(exc)
            with self.cv:
                if needs_hold:
                    self.stopped = True
                    self.hold_pending = False
                    self.phase = 'failed' if hold_error else 'held'
                    self.hold_result = {'error': hold_error} if hold_error else report
                self.busy = self.hold_pending
                if job:
                    self.results[job[0]].update(status='error' if error or hold_error else 'interrupted' if needs_hold else 'done', error=error or hold_error, stop_phase=self.phase, outcome=outcome)
                self.cv.notify_all()

    def watch(self):
        while not self.shutdown:
            with self.cv:
                expired = self.last_seen is not None and time.monotonic() - self.last_seen > self.timeout
                if expired and (not self.stopped):
                    self.stop('USB network heartbeat lost')
                elif not self.stopped:
                    try:
                        self.arm.camera.health()
                    except Exception as error:
                        self.stop('Jetson camera failure: ' + str(error))
            time.sleep(0.1)

    def close(self):
        self.stop('Receiver shutdown')
        with self.cv:
            self.cv.wait_for(lambda: not self.busy, timeout=8)
            if self.busy:
                raise RuntimeError('Stop incomplete; inspect arm and use hardware stop')
            self.shutdown = True
            self.cv.notify_all()
        self.thread.join(1)
        self.arm.close()

def make_server(engine, token, host='0.0.0.0', port=RECEIVER_PORT):

    class Handler(BaseHTTPRequestHandler):

        def log_message(self, *args):
            pass

        def answer(self, code, value):
            payload = json.dumps(value).encode('utf-8')
            self.send_response(code)
            self.send_header('Content-Type', 'application/json')
            self.send_header('Content-Length', str(len(payload)))
            self.end_headers()
            try:
                self.wfile.write(payload)
            except (BrokenPipeError, ConnectionResetError):
                pass

        def auth(self):
            if not secrets.compare_digest(self.headers.get('X-Week8-Token', ''), token):
                self.answer(401, {'error': 'Invalid token'})
                return False
            return True

        def do_GET(self):
            if not self.auth():
                return
            if self.path == '/status':
                self.answer(200, engine.status())
            elif self.path.startswith('/result/'):
                with engine.cv:
                    result = engine.results.get(self.path[len('/result/'):])
                    self.answer(200 if result else 404, dict(result) if result else {'error': 'Unknown command ID'})
            else:
                self.answer(404, {'error': 'Unknown path'})

        def do_POST(self):
            if not self.auth():
                return
            try:
                self.connection.settimeout(2)
                length = int(self.headers.get('Content-Length', '0'))
                if not 0 < length <= 2048:
                    raise ValueError('Invalid request size')
                packet = json.loads(self.rfile.read(length))
                if not isinstance(packet, dict):
                    raise ValueError('Invalid JSON object')
                if packet.get('session') != engine.session:
                    raise ValueError('Receiver session changed; reconnect')
                if self.path == '/heartbeat':
                    result = engine.heartbeat()
                elif self.path == '/stop':
                    result = engine.stop()
                elif self.path == '/clear':
                    result = engine.clear(packet.get('epoch'))
                elif self.path == '/command':
                    identifier = packet.get('id')
                    if not isinstance(identifier, str) or not 1 <= len(identifier) <= 64:
                        raise ValueError('Invalid ID')
                    result = engine.submit(identifier, packet.get('command'), packet.get('epoch'))
                else:
                    raise ValueError('Unknown path')
                self.answer(200, result)
            except (ValueError, TypeError, TimeoutError, RuntimeError, OSError) as exc:
                self.answer(409, {'error': str(exc)})
    server = ThreadingHTTPServer((host, port), Handler)
    server.daemon_threads = True
    return server


## 内置 receiver_panel

直接运行，不需要另存为 .py 文件。


In [ ]:
"""Jetson notebook controls. Starting the service connects UART but does not move."""
import copy
import html
import secrets
import threading
import time
import traceback

def create_receiver_panel(config=None):
    import ipywidgets as widgets
    from IPython.display import display
    previous = globals().get('_week8_service')
    if previous and (previous.get('engine') or previous.get('starting')):
        raise RuntimeError('旧 Week8 接收服务仍在运行；先关闭服务，等待串口释放')
    config = copy.deepcopy(SORT_CONFIG if config is None else config)
    state = {'engine': None, 'server': None, 'starting': False, 'closing': False, 'preview': None}
    globals()['_week8_service'] = state
    start = widgets.Button(description='启动 Week8 服务', button_style='success')
    stop = widgets.Button(description='停止当前动作', button_style='danger')
    close = widgets.Button(description='关闭服务')
    status = widgets.HTML(value='未启动；先关闭 Week6/Week7 的相机与舵机连接。')
    token_label = widgets.HTML(value='')
    camera = widgets.Image(format='jpeg', width=640)
    output = widgets.Output()

    def preview():
        while state['engine'] is not None and (not state['closing']):
            engine = state['engine']
            try:
                jpeg = engine.arm.camera.jpeg()
                if jpeg:
                    camera.value = jpeg
                snapshot = engine.status()
                text = 'BUSY {} / STOP {} / 阶段 {} / P4 {} / 等待空场 {}'.format(snapshot['busy'], snapshot['stopped'], snapshot['vision']['stage'], snapshot['vision']['p4_colors'], snapshot['vision']['p4_need_empty'])
                status.value = html.escape(text)
            except Exception as error:
                status.value = '预览异常：' + html.escape(str(error))
            time.sleep(0.2)

    def on_start(_):
        if state['engine'] is not None or state['starting']:
            return
        state['starting'] = True
        status.value = '正在预检、连接 SDK 和打开相机；此时不移动机械臂。'

        def worker():
            engine = None
            server = None
            try:
                engine = Engine(lambda event: Week8Arm(event, config))
                token = secrets.token_urlsafe(24)
                server = make_server(engine, token, port=RECEIVER_PORT)
                state.update(engine=engine, server=server, closing=False)
                threading.Thread(target=server.serve_forever, daemon=True).start()
                token_label.value = '<b>URL:</b> http://192.168.55.1:{}<br><b>TOKEN:</b> {}'.format(RECEIVER_PORT, html.escape(token))
                state['preview'] = threading.Thread(target=preview, daemon=True)
                state['preview'].start()
            except Exception as error:
                status.value = '启动失败：' + html.escape(str(error))
                with output:
                    print(traceback.format_exc())
                if server:
                    server.server_close()
                if engine:
                    try:
                        engine.close()
                    except Exception:
                        with output:
                            print(traceback.format_exc())
                state.update(engine=None, server=None)
            finally:
                state['starting'] = False
        threading.Thread(target=worker, daemon=True).start()

    def on_stop(_):
        if state['engine']:
            state['engine'].stop('Jetson manual stop button')

    def on_close(_):
        if state['starting']:
            status.value = '正在初始化，请等初始化完成后关闭。'
            return
        if not state['engine'] or state['closing']:
            return
        state['closing'] = True
        engine, server = (state['engine'], state['server'])
        engine.stop('Jetson service close')

        def worker():
            try:
                server.shutdown()
                server.server_close()
                engine.close()
                state.update(engine=None, server=None)
                status.value = '已关闭服务并释放相机、串口。'
                token_label.value = ''
            except Exception as error:
                status.value = '关闭失败：' + html.escape(str(error))
        threading.Thread(target=worker, daemon=True).start()
    start.on_click(on_start)
    stop.on_click(on_stop)
    close.on_click(on_close)
    display(widgets.VBox([widgets.HTML('<b>Week8 Jetson 手势选色抓取接收端</b>'), widgets.HBox([start, stop, close]), token_label, status, camera, output]))
    state['close'] = on_close
    return state


## 启动界面

点击启动 Week8 服务，把本次 TOKEN 复制到本机。地址 http://192.168.55.1:8766。仅真机模式，不需要 Jetson 安装 MediaPipe。停止当前动作按钮优先制停；关闭服务会等待制停并释放设备。


In [ ]:
service = create_receiver_panel(SORT_CONFIG)
